In [1]:
# ===============================
# 1. Imports & Drive Mount
# ===============================
import pandas as pd
import numpy as np
from google.colab import drive

drive.mount('/content/drive')

# ===============================
# 2. Load CSV
# ===============================
file_path = "/content/drive/MyDrive/GEE_Forest_TimeSeries/UP_Forest_TimeSeries_Master.csv"
df = pd.read_csv(file_path)



Mounted at /content/drive


In [2]:
print(df.head())
print(df['year'].unique())
print(df.columns)
# Keep only one row per location-year (average if needed)
# Define feature columns
features = ['BSI', 'Forest', 'NBR', 'NDMI', 'NDVI']

# Now group properly
df = df.groupby(['latitude', 'longitude', 'year'])[features].mean().reset_index()
df = df.groupby(['latitude', 'longitude', 'year'])[features].mean().reset_index()
sequence_length = 4  # 2021–2024
X_raw = []
locations = []

# Make sure duplicates are removed
df_unique = df.groupby(['latitude', 'longitude', 'year'])[features].mean().reset_index()

for (lat, lon), group in df_unique.groupby(['latitude', 'longitude']):
    group = group.sort_values('year')
    if len(group) == sequence_length:
        X_raw.append(group[features].values)
        locations.append([lat, lon])

X_raw = np.array(X_raw)       # (samples, 4, 4)
locations = np.array(locations)

print("Raw input shape:", X_raw.shape)

# ===============================
# 4. Label Creation (BEFORE SCALING)
# ===============================
# NDVI drop = vegetation loss
ndvi_drop = X_raw[:, 0, 0] - X_raw[:, -1, 0]   # 2021 - 2024
bsi_rise  = X_raw[:, -1, 3] - X_raw[:, 0, 3]   # 2024 - 2021

ndvi_thr = np.percentile(ndvi_drop, 85)
bsi_thr  = np.percentile(bsi_rise, 85)

y = ((ndvi_drop > ndvi_thr) & (bsi_rise > bsi_thr)).astype(int)

print("\nLabel distribution:")
print("No deforestation:", (y == 0).sum())
print("Deforestation:", (y == 1).sum())

# ===============================
# 5. Train–Test Split (WITH LOCATIONS)
# ===============================
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test, loc_train, loc_test = train_test_split(
    X_raw, y, locations,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)

# ===============================
# 6. Scaling (NO DATA LEAKAGE)
# ===============================
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

X_train_rs = X_train.reshape(-1, X_train.shape[-1])
X_test_rs  = X_test.reshape(-1, X_test.shape[-1])

X_train = scaler.fit_transform(X_train_rs).reshape(X_train.shape)
X_test  = scaler.transform(X_test_rs).reshape(X_test.shape)

print("Scaled train shape:", X_train.shape)

# ===============================
# 7. LSTM Model
# ===============================
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

model = Sequential([
    LSTM(64, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.Precision(name='precision')
    ]
)

model.summary()

# ===============================
# 8. Train Model (Class Weighting)
# ===============================
class_weight = {0: 1, 1: 12}

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=20,
    batch_size=128,
    class_weight=class_weight,
    verbose=1
)

# ===============================
# 9. Evaluation
# ===============================
from sklearn.metrics import classification_report

y_prob = model.predict(X_test)
y_pred = (y_prob > 0.35).astype(int)   # recall-friendly threshold

print(classification_report(y_test, y_pred))

# ===============================
# 10. Save Predictions (CORRECT LOCATIONS)
# ===============================
results_df = pd.DataFrame({
    'latitude': loc_test[:, 0],
    'longitude': loc_test[:, 1],
    'deforestation': y_pred.flatten()
})

print(results_df.head())

results_df.to_csv(
    '/content/drive/MyDrive/UP_Deforestation_Predictions.csv',
    index=False
)

print("Saved:_Deforestation_Predictions.csv")


             system:index       BSI  Forest       NBR      NDMI      NDVI  \
0  000000000000000001b6_0 -0.098624       1  0.321844  0.157256  0.457252   
1  000000000000000001b6_0  0.082582       1  0.062951 -0.033184  0.225153   
2  000000000000000001b6_0  0.111528       1  0.039800 -0.066339  0.182705   
3  000000000000000001b6_0  0.106778       1  0.039121 -0.075564  0.217314   
4  000000000000000001b7_0 -0.024777       1  0.279823  0.107181  0.352980   

   year                                               .geo  longitude  \
0  2021  {"geodesic":false,"type":"Point","coordinates"...  83.532991   
1  2022  {"geodesic":false,"type":"Point","coordinates"...  83.532991   
2  2023  {"geodesic":false,"type":"Point","coordinates"...  83.532991   
3  2024  {"geodesic":false,"type":"Point","coordinates"...  83.532991   
4  2021  {"geodesic":false,"type":"Point","coordinates"...  83.532991   

    latitude  
0  24.618780  
1  24.618780  
2  24.618780  
3  24.618780  
4  24.621474  
[2021 20

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        17,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,033 (78.25 KB)

 Trainable params: 20,033 (78.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 10s 13ms/step - accuracy: 0.3160 - loss: 1.3772 - precision: 0.1494 - recall: 0.9611 - val_accuracy: 0.8224 - val_loss: 0.3521 - val_precision: 0.4011 - val_recall: 0.9355
Epoch 2/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.8243 - loss: 0.5996 - precision: 0.4082 - recall: 0.9585 - val_accuracy: 0.8864 - val_loss: 0.2393 - val_precision: 0.5168 - val_recall: 1.0000
Epoch 3/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 7s 13ms/step - accuracy: 0.9160 - loss: 0.3094 - precision: 0.5930 - recall: 0.9781 - val_accuracy: 0.9146 - val_loss: 0.1861 - val_precision: 0.5873 - val_recall: 0.9993
Epoch 4/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.9348 - loss: 0.2429 - precision: 0.6530 - recall: 0.9858 - val_accuracy: 0.9229 - val_loss: 0.1747 - val_precision: 0.6120 - val_recall: 0.9986
Epoch 5/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.9407 - loss: 0.2230 - precision: 0.6752 - recall: 0.9825 - val_accuracy: 0.9720 - val_loss

In [3]:
!pip install folium
import pandas as pd
import folium
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
import numpy as np

# 1️⃣ Load deforestation points
df_def = pd.read_csv('/content/drive/MyDrive/UP_Deforestation_Predictions.csv')
df_deforest = df_def[df_def['deforestation']==1]

# 2️⃣ Load time-series indices
df_ts = pd.read_csv('/content/drive/MyDrive/GEE_Forest_TimeSeries/UP_Forest_TimeSeries_Master.csv')

# 3️⃣ Sort and compute changes
df_ts_sorted = df_ts.sort_values(by=['latitude','longitude','year'])

# First and last year values per point
first_year = df_ts_sorted.groupby(['latitude','longitude']).first().reset_index()
last_year  = df_ts_sorted.groupby(['latitude','longitude']).last().reset_index()

df_change = pd.merge(first_year, last_year, on=['latitude','longitude'], suffixes=('_start','_end'))

# Compute delta indices
df_change['NDVI_change'] = df_change['NDVI_end'] - df_change['NDVI_start']
df_change['NBR_change']  = df_change['NBR_end']  - df_change['NBR_start']
df_change['BSI_change']  = df_change['BSI_end']  - df_change['BSI_start']
df_change['NDMI_change'] = df_change['NDMI_end'] - df_change['NDMI_start']


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import pandas as pd
import folium
from folium.plugins import MarkerCluster

# ==============================
# THRESHOLD FUNCTION
# ==============================
def get_thresholds(series):
    series = series.dropna()
    mean = series.mean()
    std  = series.std()
    q1   = series.quantile(0.25)
    q3   = series.quantile(0.75)
    return {
        'mean': mean,
        'std': std,
        'q1': q1,
        'q3': q3,
        'lower_2std': mean - 2*std,
        'upper_2std': mean + 2*std
    }

# ==============================
# COMPUTE THRESHOLDS
# ==============================
ndvi_thresh = get_thresholds(df_change['NDVI_change'])
nbr_thresh  = get_thresholds(df_change['NBR_change'])
bsi_thresh  = get_thresholds(df_change['BSI_change'])
ndmi_thresh = get_thresholds(df_change['NDMI_change'])

print("NDVI:", ndvi_thresh)
print("NBR :", nbr_thresh)
print("BSI :", bsi_thresh)
print("NDMI:", ndmi_thresh)

# ==============================
# MERGE DATA
# ==============================
df_merged = pd.merge(
    df_deforest,
    df_change[['latitude','longitude',
               'NDVI_change','NBR_change',
               'BSI_change','NDMI_change']],
    on=['latitude','longitude'],
    how='left'
)

# ==============================
# ADAPTIVE CLASSIFIER
# ==============================
def classify_cause_adaptive(row):
    ndvi = row['NDVI_change']
    nbr  = row['NBR_change']
    bsi  = row['BSI_change']

    # Missing data protection
    if pd.isna(ndvi) or pd.isna(nbr) or pd.isna(bsi):
        return "Unknown"

    # 🔥 Fire → extreme burn signal
    if nbr < nbr_thresh['lower_2std']:
        return "Fire"

    # 🪓 Logging → vegetation drop + soil exposure
    elif ndvi < ndvi_thresh['q1'] and bsi > bsi_thresh['q3']:
        return "Logging"

    # 🌾 Agriculture → moderate change
    elif ndvi < ndvi_thresh['mean'] and bsi > bsi_thresh['q1']:
        return "Agriculture"

    # 🏙 Urban / other disturbance
    else:
        return "Urban/Other"

# ==============================
# APPLY CLASSIFICATION
# ==============================
df_merged['cause'] = df_merged.apply(classify_cause_adaptive, axis=1)

# ==============================
# SAVE OUTPUT
# ==============================
df_merged.to_csv(
    '/content/drive/MyDrive/UP_Deforestation_Causes_Adaptive.csv',
    index=False
)

# ==============================
# SUMMARY
# ==============================
print("\nDeforestation Cause Summary:")
print(df_merged['cause'].value_counts())

# ==============================
# Uttar Pradesh Map
# Approx latitude: 26–28.5, longitude: 79–83
# ==============================
m = folium.Map(location=[27.0, 80.9], zoom_start=7)

# Load UP CSV with deforestation causes
df = pd.read_csv('/content/drive/MyDrive/UP_Deforestation_Causes_Adaptive.csv')

# Print cause counts
print(df['cause'].value_counts())

# Define colors for causes
cause_colors = {
    'Fire': 'red',
    'Logging': 'orange',
    'Agriculture': 'yellow',
    'Urban/Other': 'gray'
}

# Cluster points for performance
marker_cluster = MarkerCluster().add_to(m)

# Add points to map
for _, row in df.iterrows():
    color = cause_colors.get(row['cause'], 'blue')  # default blue if cause not in dict
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=3,
        color=color,
        fill=True,
        fill_opacity=0.7,
    ).add_to(marker_cluster)

# Optional: Save map
m.save('/content/drive/MyDrive/up_adaptive.html')
print("✅ Uttar Pradesh map saved successfully!")

NDVI: {'mean': np.float64(-0.2652510239691433), 'std': 0.1000337008595929, 'q1': np.float64(-0.32452291250000004), 'q3': np.float64(-0.22724558249999996), 'lower_2std': np.float64(-0.4653184256883291), 'upper_2std': np.float64(-0.0651836222499575)}
NBR : {'mean': np.float64(-0.19241079361565383), 'std': 0.09694812323259325, 'q1': np.float64(-0.24835911250000003), 'q3': np.float64(-0.13851708499999998), 'lower_2std': np.float64(-0.3863070400808403), 'upper_2std': np.float64(0.0014854528495326647)}
BSI : {'mean': np.float64(0.10413207364711485), 'std': 0.05697352611807229, 'q1': np.float64(0.07248382938079298), 'q3': np.float64(0.13979644869189975), 'lower_2std': np.float64(-0.00981497858902973), 'upper_2std': np.float64(0.21807912588325945)}
NDMI: {'mean': np.float64(-0.09397140052149722), 'std': 0.06192138607511629, 'q1': np.float64(-0.1298248775), 'q3': np.float64(-0.05619748250000001), 'lower_2std': np.float64(-0.2178141726717298), 'upper_2std': np.float64(0.029871371628735363)}

Def

In [6]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# ==============================
# LOAD DEFORESTATION PREDICTIONS
# ==============================
df_def = pd.read_csv(
    '/content/drive/MyDrive/UP_Deforestation_Predictions.csv'
)

print("Total samples:", len(df_def))
print(df_def.head())

# Filter only deforestation = 1
df_deforest = df_def[df_def['deforestation'] == 1].copy()

print("Total deforestation samples:", len(df_deforest))

# ==============================
# LOAD CAUSE DATA
# ==============================
df_cause = pd.read_csv(
    '/content/drive/MyDrive/UP_Deforestation_Causes_Adaptive.csv'
)

print(df_cause.head())

# ==============================
# LOAD UTTAR PRADESH DISTRICTS
# ==============================
up = gpd.read_file('/content/drive/MyDrive/UP.geojson')

# CRS SAFETY
if up.crs is None:
    up.set_crs("EPSG:4326", inplace=True)

up = up.to_crs("EPSG:4326")

# Add state label (optional)
up["state"] = "Uttar Pradesh"

print(up.head())

# ==============================
# MERGE CAUSE DATA
# ==============================
df_deforest = df_deforest.merge(
    df_cause[['latitude', 'longitude', 'cause']],
    on=['latitude', 'longitude'],
    how='left'
)

print(df_deforest.head())

# ==============================
# CREATE POINT GEOMETRY
# ==============================
geometry = [
    Point(xy) for xy in zip(
        df_deforest['longitude'],
        df_deforest['latitude']
    )
]

gdf_points = gpd.GeoDataFrame(
    df_deforest,
    geometry=geometry,
    crs="EPSG:4326"
)

print(gdf_points.head())

Total samples: 12000
    latitude  longitude  deforestation
0  29.547835  79.979256              0
1  27.791629  81.892667              0
2  27.744018  82.313977              0
3  27.682035  82.405605              0
4  30.912376  78.366780              0
Total deforestation samples: 2048
    latitude  longitude  deforestation  NDVI_change  NBR_change  BSI_change  \
0  30.658153  78.699156              1     0.162568    0.061102   -0.058289   
1  29.686176  78.969549              1     0.169149   -0.043008    0.044316   
2  27.855409  81.691444              1    -0.301730   -0.140757    0.044420   
3  27.755696  82.300502              1    -0.105279   -0.036814   -0.083980   
4  25.711131  84.415136              1     0.129599    0.189243   -0.116453   

   NDMI_change        cause  
0     0.018236  Urban/Other  
1    -0.033458  Urban/Other  
2    -0.015134  Urban/Other  
3     0.064829  Urban/Other  
4     0.148105  Urban/Other  
  REMARKS_2 Country     State_Name State_Code       Dist

In [7]:
import geopandas as gpd

# ==============================
# SPATIAL JOIN (Points → Districts)
# ==============================
gdf_joined = gpd.sjoin(
    gdf_points,
    up,                # UP GeoDataFrame
    how='left',
    predicate='intersects'
)

print(gdf_joined.head())

print("Total joined points:", len(gdf_joined))
print("Points without district:", gdf_joined['Dist_Name'].isna().sum())

print(gdf_joined['Dist_Name'].value_counts())

# ==============================
# FIX DUPLICATE CAUSE COLUMNS
# ==============================
if 'cause' not in gdf_joined.columns:

    if 'cause_x' in gdf_joined.columns and 'cause_y' in gdf_joined.columns:
        gdf_joined['cause'] = gdf_joined['cause_x'].fillna(gdf_joined['cause_y'])

    elif 'cause_x' in gdf_joined.columns:
        gdf_joined['cause'] = gdf_joined['cause_x']

    elif 'cause_y' in gdf_joined.columns:
        gdf_joined['cause'] = gdf_joined['cause_y']

# Cleanup duplicate columns
gdf_joined.drop(
    columns=[c for c in ['cause_x','cause_y'] if c in gdf_joined.columns],
    inplace=True
)

# ==============================
# DISTRICT SUMMARY — Uttar Pradesh
# ==============================
district_summary = (
    gdf_joined
    .groupby('Dist_Name')
    .size()
    .reset_index(name='deforestation_points')
    .sort_values(by='deforestation_points', ascending=False)
)

print(district_summary)

print(
    "Sum of district-wise deforestation samples:",
    district_summary['deforestation_points'].sum()
)

district_summary.to_csv(
    '/content/drive/MyDrive/UP_District_Wise_Deforestation.csv',
    index=False
)

print("Saved UP district summary")

# ==============================
# DISTRICT × CAUSE SUMMARY
# ==============================
cause_summary = (
    gdf_joined
    .groupby(['Dist_Name', 'cause'])
    .size()
    .reset_index(name='count')
)

print(cause_summary.head())

cause_summary.to_csv(
    '/content/drive/MyDrive/UP_District_Wise_Deforestation_Causes.csv',
    index=False
)

print("Saved UP cause summary")

    latitude  longitude  deforestation        cause  \
0  30.658153  78.699156              1  Urban/Other   
1  29.686176  78.969549              1  Urban/Other   
2  27.855409  81.691444              1  Urban/Other   
3  27.755696  82.300502              1  Urban/Other   
4  25.711131  84.415136              1  Urban/Other   

                    geometry  index_right REMARKS_2 Country     State_Name  \
0  POINT (78.69916 30.65815)          NaN       NaN     NaN            NaN   
1  POINT (78.96955 29.68618)          NaN       NaN     NaN            NaN   
2  POINT (81.69144 27.85541)         62.0      None   India  Uttar Pradesh   
3    POINT (82.3005 27.7557)         10.0      None   India  Uttar Pradesh   
4  POINT (84.41514 25.71113)          9.0      None   India  Uttar Pradesh   

  State_Code  Dist_Name Dist_Code          state  
0        NaN        NaN       NaN            NaN  
1        NaN        NaN       NaN            NaN  
2         09  Shravasti       181  Uttar Prades

In [8]:
# =====================================================
# STEP 1: Imports
# =====================================================
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import folium
import numpy as np

# =====================================================
# STEP 2: Load UP District Boundaries
# =====================================================
up = gpd.read_file('/content/drive/MyDrive/UP.geojson')

# CRS safety
if up.crs is None:
    up.set_crs(epsg=4326, inplace=True)

up = up.to_crs(epsg=4326)
up["state"] = "Uttar Pradesh"
gdf_districts = up.copy()

# =====================================================
# STEP 3: Load Deforestation Predictions
# =====================================================
df = pd.read_csv("/content/drive/MyDrive/UP_Deforestation_Predictions.csv")

# Ensure afforestation column exists
if 'afforestation' not in df.columns:
    df['afforestation'] = 0

# =====================================================
# STEP 4: Convert to GeoDataFrame
# =====================================================
gdf_points = gpd.GeoDataFrame(
    df,
    geometry=[Point(xy) for xy in zip(df.longitude, df.latitude)],
    crs="EPSG:4326"
)

# =====================================================
# STEP 5: Spatial Join
# =====================================================
points_with_district = gpd.sjoin(
    gdf_points,
    gdf_districts,
    how="left",
    predicate="within"
)

# =====================================================
# STEP 6: District Statistics
# =====================================================
district_total = (
    points_with_district
    .groupby("Dist_Name")
    .size()
    .reset_index(name="total_samples")
)

district_deforestation = (
    points_with_district[points_with_district["deforestation"] == 1]
    .groupby("Dist_Name")
    .size()
    .reset_index(name="deforestation_samples")
)

district_afforestation = (
    points_with_district[points_with_district["afforestation"] == 1]
    .groupby("Dist_Name")
    .size()
    .reset_index(name="afforestation_samples")
)

# Merge
gdf_districts = gdf_districts.merge(district_total, on="Dist_Name", how="left")
gdf_districts = gdf_districts.merge(district_deforestation, on="Dist_Name", how="left")
gdf_districts = gdf_districts.merge(district_afforestation, on="Dist_Name", how="left")

gdf_districts.fillna(0, inplace=True)

# =====================================================
# STEP 7: Area + Rate Calculations
# =====================================================
PIXEL_AREA_SQKM = 0.0001
PIXEL_AREA_HA = 0.01

gdf_districts["deforestation_rate_%"] = (
    gdf_districts["deforestation_samples"] /
    gdf_districts["total_samples"] * 100
).fillna(0)

gdf_districts["deforestation_area_sqkm"] = (
    gdf_districts["deforestation_samples"] * PIXEL_AREA_SQKM
)

gdf_districts["deforestation_area_ha"] = (
    gdf_districts["deforestation_samples"] * PIXEL_AREA_HA
)

gdf_districts["afforestation_rate_%"] = (
    gdf_districts["afforestation_samples"] /
    gdf_districts["total_samples"] * 100
).fillna(0)

gdf_districts["afforestation_area_sqkm"] = (
    gdf_districts["afforestation_samples"] * PIXEL_AREA_SQKM
)

gdf_districts["afforestation_area_ha"] = (
    gdf_districts["afforestation_samples"] * PIXEL_AREA_HA
)

# =====================================================
# STEP 8: Create Folium Map (center UP)
# =====================================================
m = folium.Map(location=[26.85, 80.95], zoom_start=6)

# =====================================================
# STEP 9: Choropleth
# =====================================================
min_val = gdf_districts["deforestation_samples"].min()
max_val = gdf_districts["deforestation_samples"].max()

if max_val == min_val:
    bins = [min_val, max_val + 1]
else:
    bins = np.linspace(min_val, max_val, num=6).tolist()

folium.Choropleth(
    geo_data=gdf_districts.to_json(),
    data=gdf_districts,
    columns=["Dist_Name", "deforestation_samples"],
    key_on="feature.properties.Dist_Name",
    fill_color="YlOrRd",
    bins=bins,
    fill_opacity=0.7,
    line_opacity=0.4,
    legend_name="Deforestation Samples (Uttar Pradesh)"
).add_to(m)

# =====================================================
# STEP 10: Tooltips
# =====================================================
folium.GeoJson(
    gdf_districts,
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "Dist_Name",
            "deforestation_samples",
            "deforestation_rate_%",
            "deforestation_area_sqkm",
            "deforestation_area_ha",
            "afforestation_samples",
            "afforestation_rate_%",
            "afforestation_area_sqkm",
            "afforestation_area_ha"
        ],
        aliases=[
            "District:",
            "Deforestation samples:",
            "Deforestation rate (%):",
            "Deforestation Area (sq.km):",
            "Deforestation Area (ha):",
            "Afforestation samples:",
            "Afforestation rate (%):",
            "Afforestation Area (sq.km):",
            "Afforestation Area (ha):"
        ],
        localize=True
    )
).add_to(m)

# =====================================================
# STEP 11: Alert Popup
# =====================================================
state_def = gdf_districts["deforestation_samples"].sum()

top3 = (
    gdf_districts.sort_values(by="deforestation_samples", ascending=False)
    .head(3)
)

top_districts_html = ""
for _, row in top3.iterrows():
    top_districts_html += f"• {row['Dist_Name']} ({int(row['deforestation_samples'])})<br>"

popup_html = f"""
<b>🌳 UP Afforestation Alert</b><br><br>
Total Deforestation Points: <b>{int(state_def)}</b><br><br>
High Impact Districts:<br>
{top_districts_html}<br>
🌱 Immediate afforestation drives recommended.
"""

folium.Marker(
    location=[26.85, 80.95],
    popup=popup_html,
    icon=folium.Icon(color="green")
).add_to(m)

# =====================================================
# STEP 12: Save Map
# =====================================================
folium.LayerControl().add_to(m)
m.save("/content/drive/MyDrive/up_forest.html")

print("✅ UP map saved successfully with afforestation recommendation!")

/tmp/ipykernel_186/636044930.py:80: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  gdf_districts.fillna(0, inplace=True)


✅ UP map saved successfully with afforestation recommendation!
